# 01 — Data Inventory and Validated WR Cohort

This notebook inventories all nine official competition files and builds the reproducible 2023–2025 wide-receiver cohort used by downstream reception and YAC analysis. It validates player identifiers and draft years, reports tracking/Combine availability, and writes `data/processed/clean_wr_players.csv`.

**Google Colab:** clone the repository, upload or mount the official Kaggle files in a private data directory, and set `NFL_DATA_DIR` below. Do not commit raw competition files.

In [ ]:
# Run this cell only in a fresh Google Colab session.
# !git clone https://github.com/k1lst1x/nfl-big-data-bowl-2027.git
# %cd nfl-big-data-bowl-2027
# !pip install -r requirements.txt

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / 'src' / 'nfl_bdb').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from nfl_bdb.data import FILES, coverage_report, file_inventory
from nfl_bdb.players import cohort_summary, write_wr_cohort

## Configure the official Kaggle data location

The directory must contain `players.csv`, `combine_results.csv`, and the Combine player-tracking files expected by the repository inventory. In Colab, this can point to a mounted Google Drive directory such as `/content/drive/MyDrive/nfl-bdb-2027`.

In [ ]:
default_data_dir = repo_root / 'data' / 'raw' / 'nfl-big-data-bowl-2027'
if not default_data_dir.exists():
    default_data_dir = repo_root / 'data' / 'raw'
NFL_DATA_DIR = Path(os.environ.get('NFL_DATA_DIR', default_data_dir))
OUTPUT = repo_root / 'data' / 'processed' / 'clean_wr_players.csv'

required = list(FILES)
missing = [name for name in required if not (NFL_DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        f'Missing {missing} in {NFL_DATA_DIR}. Set NFL_DATA_DIR to the official Kaggle data directory.'
    )

print(f'Data directory: {NFL_DATA_DIR}')
print(f'Output: {OUTPUT}')

In [ ]:
inventory = file_inventory(NFL_DATA_DIR)
display(inventory[['file', 'rows', 'columns', 'size_mb']])
expected_rows = {
    'players.csv': 510, 'combine_results.csv': 510,
    'combine_tracking.csv': 463189, 'player_career_successes.csv': 510,
    'player_play.csv': 314197, 'games.csv': 1002,
    'game_tracking_2023.csv': 3665443,
    'game_tracking_2024.csv': 7827944,
    'game_tracking_2025.csv': 11160903,
}
assert inventory['file'].tolist() == list(FILES)
assert inventory.set_index('file')['rows'].to_dict() == expected_rows

schema_rows = []
for name in FILES:
    sample = pd.read_csv(NFL_DATA_DIR / name, nrows=1000)
    schema_rows.extend(
        {'file': name, 'column': column, 'inferred_dtype': str(dtype)}
        for column, dtype in sample.dtypes.items()
    )
schema = pd.DataFrame(schema_rows)
display(schema)

In [ ]:
reports = coverage_report(NFL_DATA_DIR)
display(reports['summary'])
display(reports['by_draft_year'])
display(reports['diagnostics'])
display(reports['traditional_missingness'])
display(reports['game_periods'])

diagnostics = reports['diagnostics'].set_index('check')['count']
zero_checks = [
    'players: missing nfl_id',
    'players: rows in duplicate nfl_id groups',
    'combine results: missing nfl_id',
    'combine results: unique IDs absent from players',
    'combine results: rows in duplicate nfl_id + draft_year groups',
    'PLAYER tracking: missing nfl_id',
    'PLAYER tracking: unique IDs absent from players',
    'player plays: missing nfl_id',
    'player plays: unique IDs absent from players',
    'games: missing game_id',
    'player plays: rows in duplicate game_id + play_id + nfl_id groups',
    'player plays: unmatched or missing game_id',
]
assert diagnostics[zero_checks].eq(0).all()
summary_controls = reports['summary'].set_index('measure')['count']
assert summary_controls['Players with an ID'] == 510
assert summary_controls['WR with an ID'] == 106
assert summary_controls['Candidate receiving rows (unvalidated)'] == 4034
assert diagnostics['candidate rows marked target=False'] == 2

In [ ]:
cohort = write_wr_cohort(NFL_DATA_DIR, OUTPUT)
summary = cohort_summary(cohort)
display(summary)
display(cohort.head())

In [ ]:
expected_counts = {2023: 38, 2024: 31, 2025: 37}
actual_counts = cohort.groupby('draft_year').size().to_dict()

assert cohort['nfl_id'].is_unique, 'nfl_id must be unique'
assert cohort['nfl_position'].eq('WR').all(), 'cohort must contain only WRs'
assert actual_counts == expected_counts, (
    f'Expected {expected_counts}, found {actual_counts}; inspect source coverage before continuing.'
)
assert len(cohort) == 106
print(f'Validated {len(cohort)} WRs and wrote {OUTPUT}')